# Representation Engineering and Steering

**Survey Reference:** Section 12.3 - Representation Engineering and Steering

## Overview

**Representation engineering** uses insights about model internals to control behavior. The key technique is **steering vectors**: adding carefully chosen vectors to activations to shift model behavior.

### Applications

- Increase/decrease honesty, helpfulness, refusal
- Control sentiment, style, persona
- Mitigate biases
- Induce or suppress capabilities

## Learning Objectives

1. Extract steering vectors from contrastive examples
2. Apply steering vectors at inference time
3. Understand steering strength and layer selection
4. Evaluate steering effectiveness and side effects

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Extracting Steering Vectors

The simplest method: **difference in means** between contrastive examples.

In [ ]:
# Load model
model_name = "gpt2-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def get_hidden_state(model, tokenizer, text, layer_idx, position=-1):
    """Get hidden state at a specific layer and position."""
    inputs = tokenizer(text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    return outputs.hidden_states[layer_idx][0, position, :]

def extract_steering_vector(
    model,
    tokenizer,
    positive_examples: list[str],
    negative_examples: list[str],
    layer_idx: int,
) -> torch.Tensor:
    """
    Extract steering vector using difference in means.
    
    steering_vector = mean(positive) - mean(negative)
    """
    # Collect positive representations
    positive_reps = []
    for text in positive_examples:
        h = get_hidden_state(model, tokenizer, text, layer_idx)
        positive_reps.append(h)
    positive_mean = torch.stack(positive_reps).mean(dim=0)
    
    # Collect negative representations
    negative_reps = []
    for text in negative_examples:
        h = get_hidden_state(model, tokenizer, text, layer_idx)
        negative_reps.append(h)
    negative_mean = torch.stack(negative_reps).mean(dim=0)
    
    # Steering vector
    return positive_mean - negative_mean

In [ ]:
# Example: Sentiment steering
positive_examples = [
    "I am feeling very happy and joyful today!",
    "This is wonderful, I love it so much!",
    "Everything is going great, life is beautiful!",
    "I'm so excited about this amazing opportunity!",
]

negative_examples = [
    "I am feeling very sad and miserable today.",
    "This is terrible, I hate it so much.",
    "Everything is going badly, life is hard.",
    "I'm so worried about this awful situation.",
]

# Extract steering vector from middle layer
layer_idx = model.config.n_layer // 2
sentiment_vector = extract_steering_vector(
    model, tokenizer, positive_examples, negative_examples, layer_idx
)

print(f"Steering vector shape: {sentiment_vector.shape}")
print(f"Steering vector norm: {sentiment_vector.norm():.4f}")

## 2. Applying Steering Vectors

In [ ]:
class SteeringHook:
    """
    Hook to add steering vector to activations.
    """
    
    def __init__(self, steering_vector: torch.Tensor, strength: float = 1.0):
        self.steering_vector = steering_vector
        self.strength = strength
    
    def __call__(self, module, input, output):
        hidden_states = output[0]
        # Add steering vector to all positions
        hidden_states = hidden_states + self.strength * self.steering_vector
        return (hidden_states,) + output[1:]


def generate_with_steering(
    model,
    tokenizer,
    prompt: str,
    steering_vector: torch.Tensor,
    layer_idx: int,
    strength: float = 1.0,
    max_new_tokens: int = 50,
) -> str:
    """
    Generate text with steering vector applied.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Register hook
    layer = model.transformer.h[layer_idx]
    hook = SteeringHook(steering_vector, strength)
    handle = layer.register_forward_hook(hook)
    
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        handle.remove()
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# Test steering
prompt = "I went to the store today and"

print("=" * 50)
print("Without steering:")
print("=" * 50)
with torch.no_grad():
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=50, do_sample=True, 
                             temperature=0.7, pad_token_id=tokenizer.eos_token_id)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

print("\n" + "=" * 50)
print("With POSITIVE steering:")
print("=" * 50)
result_positive = generate_with_steering(
    model, tokenizer, prompt, sentiment_vector, layer_idx, strength=2.0
)
print(result_positive)

print("\n" + "=" * 50)
print("With NEGATIVE steering:")
print("=" * 50)
result_negative = generate_with_steering(
    model, tokenizer, prompt, -sentiment_vector, layer_idx, strength=2.0
)
print(result_negative)

## 3. Steering Strength Analysis

Too little steering: no effect. Too much: breaks coherence.

In [ ]:
def analyze_steering_strength(
    model,
    tokenizer,
    prompt: str,
    steering_vector: torch.Tensor,
    layer_idx: int,
    strengths: list[float],
):
    """
    Analyze effect of different steering strengths.
    """
    results = []
    for strength in strengths:
        output = generate_with_steering(
            model, tokenizer, prompt, steering_vector, layer_idx, 
            strength=strength, max_new_tokens=30
        )
        results.append({
            'strength': strength,
            'output': output,
        })
    return results

# Analyze different strengths
strengths = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0]
# results = analyze_steering_strength(model, tokenizer, prompt, sentiment_vector, layer_idx, strengths)

## 4. Layer Selection

Which layer to steer matters!

In [ ]:
def compare_steering_layers(
    model,
    tokenizer,
    positive_examples: list[str],
    negative_examples: list[str],
    test_prompt: str,
    strength: float = 1.0,
):
    """
    Compare steering at different layers.
    """
    n_layers = model.config.n_layer
    layers_to_test = [0, n_layers//4, n_layers//2, 3*n_layers//4, n_layers-1]
    
    results = []
    for layer_idx in layers_to_test:
        vector = extract_steering_vector(
            model, tokenizer, positive_examples, negative_examples, layer_idx
        )
        output = generate_with_steering(
            model, tokenizer, test_prompt, vector, layer_idx, 
            strength=strength, max_new_tokens=30
        )
        results.append({
            'layer': layer_idx,
            'output': output,
        })
        print(f"Layer {layer_idx}: {output[:100]}...")
    
    return results

# compare_steering_layers(model, tokenizer, positive_examples, negative_examples, prompt)

## 5. Advanced: Multi-Layer Steering

In [ ]:
def generate_with_multilayer_steering(
    model,
    tokenizer,
    prompt: str,
    layer_vectors: dict[int, torch.Tensor],  # layer_idx -> vector
    strength: float = 1.0,
    max_new_tokens: int = 50,
):
    """
    Apply steering vectors at multiple layers.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    handles = []
    
    for layer_idx, vector in layer_vectors.items():
        layer = model.transformer.h[layer_idx]
        hook = SteeringHook(vector, strength)
        handle = layer.register_forward_hook(hook)
        handles.append(handle)
    
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        for handle in handles:
            handle.remove()
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

## 6. Evaluating Steering

How to measure if steering worked without breaking the model?

In [ ]:
def evaluate_steering(
    model,
    tokenizer,
    prompts: list[str],
    steering_vector: torch.Tensor,
    layer_idx: int,
    strength: float,
):
    """
    Evaluate steering on multiple prompts.
    
    Measures:
    - Perplexity (coherence)
    - Target behavior (if measurable)
    """
    # Measure perplexity with and without steering
    # Lower perplexity = more coherent
    
    # For sentiment, could use sentiment classifier on outputs
    
    pass

# TODO: Implement evaluation

## Key Takeaways

1. **Steering vectors** = difference in means between contrastive examples
2. **Layer selection matters:** Mid-to-late layers often work best
3. **Strength tuning:** Too weak = no effect, too strong = breaks coherence
4. **Multi-layer steering** can be more effective
5. **Evaluation is hard:** Need to balance target behavior vs. coherence

## Next Steps

→ **08_safety_applications/01_concept_erasure.ipynb**: Remove unwanted concepts from models